# Plot all runs

Script to plot results from all 6 original runs together.

In [ ]:
import logging
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.WARNING)

In [ ]:
# SETTINGS

scenarios=['base_2025_correct', 'base_NZE_2025_correct', 'sudden_2025_correct', 'sudden_NZE_2025_correct', 'gradual_2025_correct', 'gradual_NZE_2025_correct']
labels = ['BASE', 'BASE NZE', 'SC', 'SC NZE', 'GC', 'GC NZE']

base_file_name = 'base_2025_correct'
sudden_file_name = 'sudden_2025_correct'
gradual_file_name = 'gradual_2025_correct'
base_NZE_file_name = 'base_NZE_2025_correct'
sudden_NZE_file_name = 'sudden_NZE_2025_correct'
gradual_NZE_file_name = 'gradual_NZE_2025_correct'

path = '../result_data/'
fig_path = '../figures/'

years = [2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040]
first_year = years[0]
final_year = years[-1]

In [ ]:
# COLORS

red1 = '#891D2D'
red2 = '#BA3B31'
orange = '#F58221'
yellow = '#FCAF19'
brown = '#440A15'
brown2 = '#B45419'
purple1 = '#3B1053'
purple2 = '#76518E'
purple3 = '#B69DC7'
teal1 = '#032838'
teal2 = '#154655'
teal3 = '#527D77'
teal4 = '#8DB5AF'
teal1 = '#294839'
green1 = '#6DA08C'
green2 = '#6E966E'
green3 = '#A3BDA3'
beige1 = '#7A693B'
beige2 = '#A89677'
beige3 = '#D2CDAD'
grey1 = '#E7E7E7'
grey2 = '#D7D7D7'
grey3 = '#C6C6C6'
grey4 = '#939393'
blue1 = '#3EA1C0'

In [ ]:
# Helper functions

def get_colors(carriers):
    colors = [beige2, beige3, teal3, beige1, teal4, yellow, teal2, brown, brown, brown2, grey4, grey1]
    names = ['CCGT',    'OCGT',  'Biomass',   'Oil',  'Wind',  'Solar'  ,'Hydro', 'Battery', 'Nbattery', 'Geothermal', 'Lost load', 'Demand']
    color_dict = dict(zip(names, colors))
    colors_new = [color_dict[carrier] for carrier in carriers]
    return colors_new    


In [ ]:
# PLOT PV OF COSTS

yearly_costs_all = pd.read_csv(path + 'total_costs.csv')
capcost = pd.read_csv(path + 'capcost.csv')

yearly_costs = yearly_costs_all[scenarios]
total_costs = yearly_costs.sum(axis=0) /1000
total_costs.columns = labels

costs_dict = {label: total_costs[i] for i, label in enumerate(yearly_costs.columns)}
costs_df = pd.DataFrame([costs_dict])

years = list(range(first_year, 2041))  # From year 2023 to 2037
yearly_costs.index = years

yearly_costs.columns = labels

colors = [brown2, teal4, beige2, teal3, beige3, teal2]

plt.figure(figsize=(12, 4))  # Adjust the size as needed
for column in yearly_costs.columns:
    yearly_costs[column].plot(label=column, color=colors[yearly_costs.columns.get_loc(column)], linewidth=2, fontsize=14)
plt.xlabel('Year', fontsize=18)
plt.ylabel('PV [million €]', fontsize=18)
plt.legend( loc='upper center', bbox_to_anchor=(0.5, -0.19), ncol=3, fontsize=14)
plt.grid(axis='y')
plt.ylim(0,500)
plt.xlim(first_year,final_year)
plt.savefig(fig_path + 'pv.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT NPV AND SUBSIDIES

subsidies_all = pd.read_csv(path + 'subsidies.csv')
subsidies = subsidies_all[scenarios]
subsidies = subsidies / 1000
print(subsidies)


# Create a new figure and axis for the plot
fig, ax = plt.subplots(figsize=(12, 4))

# The x-axis positions for each bar
index = np.arange(len(costs_df.columns))
colors = [brown2, teal4, beige2, teal3, beige3, teal2]

total_costs_bars = ax.bar(index, costs_df.iloc[0], label='NPV', color=colors, edgecolor=colors)

fill_colors_with_alpha = [mcolors.to_rgba(color, alpha=0.4) for color in colors]  # Fill colors with transparency
solid_edge_colors = [mcolors.to_rgba(color, alpha=1) for color in colors]  # Edge colors without transparency

# Plotting the subsidies bars on top of the total costs (NPV)
subsidies_bars = ax.bar(index, subsidies.iloc[0], label='Subsidies', bottom=costs_df.iloc[0], 
                        color=fill_colors_with_alpha, edgecolor=solid_edge_colors, linestyle='--', linewidth=2)

ax.set_xticks(index, fontsize=14)
ax.set_xticklabels(labels, rotation=0)

# # Set the y-axis label
npv_patch = mpatches.Patch(color='darkgrey', label='NPV')
subsidies_patch = mpatches.Patch(facecolor=grey1, edgecolor='darkgrey', linestyle='--', linewidth=1, label='Subsidies')

# Create the legend with custom patches
plt.legend(handles=[subsidies_patch, npv_patch], loc='upper center', edgecolor=grey1, bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=14)
plt.ylabel('Cost [billion €]', fontsize=18)


# Add a title and a legend

plt.ylim(0,8)
plt.grid(axis='y', color=grey1)

# Adjust layout and show the plot
plt.tight_layout()
plt.savefig(fig_path + 'costs.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT EMISSIONS

emissions_all = pd.read_csv(path + 'emissions.csv')
emissions = emissions_all[scenarios]

years = list(range(first_year, final_year+1))  # From year 2023 to 2037
emissions.index = years

emissions.columns = labels
colors = [brown2, teal4, beige2, teal3, beige3, teal2]

plt.figure(figsize=(12, 4))  # Adjust the size as needed
for column in emissions.columns:
    emissions[column].plot(label=column, color=colors[emissions.columns.get_loc(column)], linewidth=2, fontsize=14)

plt.xlabel('Year', fontsize=18)
plt.ylabel('Emissions [Mt CO2 eqiuivalent]', fontsize=18)
plt.legend( loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=14)
plt.grid(axis='y')
plt.ylim(0,8)
plt.xlim(first_year,final_year)
plt.savefig(fig_path + 'emissions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT FINAL CAPACITY

labels.insert(0, '2024')

base_start_capacity = pd.read_csv(path + base_file_name + '_24_final_capacity.csv', index_col=0)
base_final_capacity = pd.read_csv(path + base_file_name + '_final_capacity.csv', index_col=0)
base_final_capacity_cap = pd.read_csv(path + base_NZE_file_name + '_final_capacity.csv', index_col=0)
sudden_final_capacity = pd.read_csv(path + sudden_file_name + '_final_capacity.csv', index_col=0)
sudden_final_capacity_cap = pd.read_csv(path + sudden_NZE_file_name + '_final_capacity.csv', index_col=0)
gradual_final_capacity = pd.read_csv(path + gradual_file_name + '_final_capacity.csv', index_col=0)
gradual_final_capacity_cap = pd.read_csv(path + gradual_NZE_file_name + '_final_capacity.csv', index_col=0)

desired_order = ['CCGT', 'OCGT', 'Oil', 'Geothermal', 'Hydro', 'Wind', 'Solar', 'Biomass']

all_data = pd.concat([base_start_capacity, base_final_capacity,base_final_capacity_cap, sudden_final_capacity, sudden_final_capacity_cap,  gradual_final_capacity,gradual_final_capacity_cap], axis=0)
all_data = all_data[desired_order]
all_data.index = labels

ax = all_data.plot(kind='bar', stacked=True, figsize=(10, 4), color=get_colors(all_data.columns), legend=False, zorder=3, fontsize=14)   
plt.setp(ax.get_xticklabels(), rotation=0)
ax.set_ylabel('Installed Capacity [MW]', fontsize=18)   
ax.legend(title='Capacity Type')
plt.legend( loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=4, fontsize=14)
plt.grid(axis='y', zorder=0, color=grey1)

plt.savefig(fig_path + 'final_capacity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT NEW CAPACITY

labels = labels[1:]  # Remove '2024' from labels for the new capacity plot 

new_capacity_base = pd.read_csv(path + base_file_name + '_new_capacity.csv', index_col=0)
new_capacity_sudden = pd.read_csv(path + sudden_file_name + '_new_capacity.csv', index_col=0)
new_capacity_gradual = pd.read_csv(path + gradual_file_name + '_new_capacity.csv', index_col=0)
new_capacity_base_cap = pd.read_csv(path + base_NZE_file_name + '_new_capacity.csv', index_col=0)
new_capacity_sudden_cap = pd.read_csv(path + sudden_NZE_file_name + '_new_capacity.csv', index_col=0)
new_capacity_gradual_cap = pd.read_csv(path + gradual_NZE_file_name + '_new_capacity.csv', index_col=0)

fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)

axes = axes.flatten()

new_capacity_scenarios = [
    ('BASE', new_capacity_base),
    ('BASE NZE', new_capacity_base_cap),
    ('SC', new_capacity_sudden),
    ('SC NZE', new_capacity_sudden_cap),
    ('GC', new_capacity_gradual),
    ('GC NZE', new_capacity_gradual_cap),

]


for ax, (title, capacity_data) in zip(axes, new_capacity_scenarios):
    capacity_data.plot.bar(stacked=True, ax=ax, color=get_colors(capacity_data.columns), legend=False, zorder=3, fontsize=18)
    ax.set_xlabel('')
    ax.set_xticks(range(len(capacity_data.index)))
    ax.set_xticklabels(capacity_data.index, rotation=90)
    ax.set_title(title, fontsize=16)
    ax.set_ylim(0, 450)
    ax.grid(axis='y', zorder=0)
    
# Only show y-ticks in left column
for i, ax in enumerate(axes):
    if i % 2 == 1:  # høyre kolonne
        ax.set_yticklabels([])
        ax.set_ylabel('')

# Set the x-axis label on the last subplot
fig.text(0.5, 0.01, 'Year', ha='center', fontsize=18)
fig.text(0.04, 0.5, 'Additional Installed Capacity [MW]', va='center', rotation='vertical', fontsize=18)

# Create a single shared legend
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.01), ncol=4, fontsize=14)

# Adjust the layout
plt.subplots_adjust(hspace=0.3, top=0.9, bottom=0.15, wspace=0.08)
plt.savefig(fig_path + 'new_capacity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT POWER PRODUCTION

production_base = pd.read_csv(path + base_file_name + '_production.csv', index_col=0)
production_sudden = pd.read_csv(path + sudden_file_name + '_production.csv', index_col=0)
production_gradual = pd.read_csv(path + gradual_file_name + '_production.csv', index_col=0)
production_base_cap = pd.read_csv(path + base_NZE_file_name + '_production.csv', index_col=0)
production_sudden_cap = pd.read_csv(path + sudden_NZE_file_name + '_production.csv', index_col=0)
production_gradual_cap = pd.read_csv(path + gradual_NZE_file_name + '_production.csv', index_col=0)

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True,
    gridspec_kw={'wspace': 0.1, 'hspace': 0.3})  # Adjust figsize as needed

# Flatten the 2D array of axes to 1D for easier iteration
axes = axes.flatten()

# Define your scenarios
scenarios = [
    (production_base, 'BASE'),
    (production_base_cap, 'BASE NZE'),
    (production_sudden, 'SC'),
    (production_sudden_cap, 'SC NZE'),
    (production_gradual, 'GC'),
    (production_gradual_cap, 'GC NZE')
]


# Loop through the scenarios to create each subplot
for ax, (production_data, title) in zip(axes, scenarios):
    # Ensure production_data is a DataFrame and not a numpy array
    if isinstance(production_data, pd.DataFrame):
        production_data.index = production_data.index.astype(str).str.strip().astype(int)

        ax.stackplot(production_data.index, production_data.T, 
                     colors=get_colors(production_data.columns), labels=production_data.columns)
        ax.set_ylim(0, 22)
        #ax.set_xlim(first_year, last_year)
        ax.set_title(title, fontsize=18)
        # rotate x-tick labels for better readability including all years
        ax.set_xlim(production_data.index.min(), production_data.index.max())
        xticks = production_data.index[::5]  # hvert 5. element i indeksen
        ax.set_xticks(xticks + 1)
        ax.set_xticklabels(xticks + 1, fontsize=18)
        #ax.tick_params(axis='both', which='major', labelsize=14) 
        ax.tick_params(axis='y', labelsize=18)
    else:
        print(f"Data for {title} is not in DataFrame format.")

# Fjern y-ticks i høyre kolonne
for i, ax in enumerate(axes):
    if i % 2 == 1:  # høyre kolonne
        ax.set_yticklabels([])
        ax.set_ylabel('')

# Set the shared x and y-axis labels
fig.text(0.5, 0.04, 'Year', ha='center', fontsize=18)
fig.text(0.04, 0.5, 'Power Generation [TWh]', va='center', rotation='vertical', fontsize=18)

colors=[beige2,beige3,beige1,brown2, teal2, teal4,yellow,teal3, brown, grey4, grey1]
labels = [label for label in production_base.columns if label != 'Nbattery']  # Exclude 'Nbattery'
patches = [mpatches.Patch(color=color, label=label) for label, color in zip(labels, colors) if label != 'Nbattery']
handles = [mpatches.Patch(color=color, label=label) for label, color in zip(labels, colors)]
fig.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=4, fontsize=14)

#plt.subplots_adjust(hspace=0.3, bottom=0.15)

# Save the figure
plt.savefig(fig_path + 'power_prod.png', dpi=300, bbox_inches='tight') 
plt.show()

In [ ]:
# PLOT CAPACITY FACTORS

base_capacity_factor = pd.read_csv(path + base_file_name + '_capacity_factors.csv', index_col=0)
sudden_capacity_factor = pd.read_csv(path + sudden_file_name + '_capacity_factors.csv', index_col=0)
gradual_capacity_factor = pd.read_csv(path + gradual_file_name + '_capacity_factors.csv', index_col=0)
base_capacity_factor_cap = pd.read_csv(path + base_NZE_file_name + '_capacity_factors.csv', index_col=0)
sudden_capacity_factor_cap = pd.read_csv(path + sudden_NZE_file_name + '_capacity_factors.csv', index_col=0)
gradual_capacity_factor_cap = pd.read_csv(path + gradual_NZE_file_name + '_capacity_factors.csv', index_col=0)

desired_order = ['CCGT', 'OCGT', 'Oil', 'Geothermal', 'Hydro', 'Wind', 'Solar', 'Biomass']
base_capacity_factor.columns = ['CCGT', 'OCGT', 'Biomass', 'Geothermal', 'Hydro', 'Oil', 'Wind', 'Solar']
sudden_capacity_factor.columns = ['CCGT', 'OCGT', 'Biomass', 'Geothermal', 'Hydro', 'Oil', 'Wind', 'Solar']
gradual_capacity_factor.columns = ['CCGT', 'OCGT', 'Biomass', 'Geothermal', 'Hydro', 'Oil', 'Wind', 'Solar']
base_capacity_factor_cap.columns = ['CCGT', 'OCGT', 'Biomass', 'Geothermal', 'Hydro', 'Oil', 'Wind', 'Solar']
sudden_capacity_factor_cap.columns = ['CCGT', 'OCGT', 'Biomass', 'Geothermal', 'Hydro', 'Oil', 'Wind', 'Solar']
gradual_capacity_factor_cap.columns = ['CCGT', 'OCGT', 'Biomass', 'Geothermal', 'Hydro', 'Oil', 'Wind', 'Solar']

scenarios = {
    'BASE': base_capacity_factor,
    'BASE NZE': base_capacity_factor_cap,
    'SC': sudden_capacity_factor,
    'SC NZE': sudden_capacity_factor_cap,
    'GC': gradual_capacity_factor,
    'GC NZE': gradual_capacity_factor_cap
}


colors = get_colors(desired_order)  # Replace this with your actual function for color mapping

# Create subplots - one for each scenario
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 10), sharex=True, sharey=True)
axes = axes.flatten()  # Flatten if needed

used_labels = set()

# Plot each scenario in its own subplot
for ax, (title, costs_df) in zip(axes, scenarios.items()):
    for carrier, color in zip(desired_order, colors):
        if carrier in costs_df.columns:
            ax.plot(costs_df.index, costs_df[carrier], label=carrier, color=color, marker='o')
            used_labels.add(carrier)
        else:
            print(f"'{carrier}' not found in '{title}' scenario")
    ax.set_ylim(0,1)
    ax.set_xlim(first_year,final_year)
    ax.set_title(title, fontdict={'fontsize':18})
    ax.grid(axis='y', color=grey2)
    # set x-ticks every 5 years
    xticks = list(range(first_year + 1, final_year + 1, 5))
    ax.set_xticks(xticks)
    # set font size for x and y ticks
    ax.tick_params(axis='both', which='major', labelsize=18)

# Set common labels and title
fig.text(0.5, 0.04, 'Year', ha='center', fontsize=18)
fig.text(0.04, 0.5, 'Capacity Factor', va='center', rotation='vertical', fontsize=18)

# Only include used labels in the legend
handles = [mpatches.Patch(color=color, label=label) for label, color in zip(desired_order, colors) if label in used_labels]
fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=4, fontsize=14)
plt.subplots_adjust(wspace=0.1)
plt.savefig(fig_path + 'capacity_factors.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT DISPATCH

snapshot_base = pd.read_csv(path + base_file_name + '_snapshots.csv', index_col=0, parse_dates=True)
snapshot_sudden = pd.read_csv(path + sudden_file_name + '_snapshots.csv', index_col=0, parse_dates=True)
snapshot_gradual = pd.read_csv(path + gradual_file_name + '_snapshots.csv', index_col=0, parse_dates=True)
snapshot_base_cap = pd.read_csv(path + base_NZE_file_name + '_snapshots.csv', index_col=0, parse_dates=True)
snapshot_sudden_cap = pd.read_csv(path + sudden_NZE_file_name + '_snapshots.csv', index_col=0, parse_dates=True)
snapshot_gradual_cap = pd.read_csv(path + gradual_NZE_file_name + '_snapshots.csv', index_col=0, parse_dates=True)

first_date = "2013-12-28"
second_date = "2013-12-31"
start_date = pd.to_datetime(first_date)
end_date = pd.to_datetime(second_date)

# Create the subplots with 2 columns and 3 rows
fig, axes = plt.subplots(3, 2, figsize=(12,10), sharex=True, sharey=True)  # Adjust figsize as needed

# Flatten the axes array for easy iteration
axes = axes.flatten()

# Define your scenarios
scenarios = [
    ('BASE', snapshot_base),
    ('BASE NZE', snapshot_base_cap),  # Assuming you have this data
    ('SC', snapshot_sudden),
    ('SC NZE', snapshot_sudden_cap),  # Assuming you have this data
    ('GC', snapshot_gradual),
    ('GC NZE', snapshot_gradual_cap)  # Assuming you have this data
]

# Loop through the scenarios to create each subplot
for ax, (title, data) in zip(axes, scenarios):
    # Plot the data using stackplot or any other plotting function
    ax.stackplot(data.index, data['CCGT'], data['OCGT'], data['Oil'], data['Geothermal'], data['Hydro'],
                 data['Wind'], data['Solar'], data['Biomass'], data['Battery'], data['Lost load'],
                 colors=[beige2,beige3,beige1,brown2, teal2, teal4,yellow,teal3, brown, grey4], zorder=2)
    ax.stackplot(data.index, data['Demand'], data['Nbattery'], colors=[grey1, brown], zorder=2)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(start_date, end_date)
    ax.grid(axis='y', color=grey2)
    ax.set_ylim(-3500,3500)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
    ax.tick_params(axis='x', labelrotation=90, labelsize=18)
    ax.tick_params(axis='both', which='major', labelsize=18) 
    ax.set_yticks([-3000, -1500, 0, 1500, 3000])


# Set common labels
#fig.text(0.5, 0.04, 'Date', ha='center', va='center', fontsize=18)
fig.text(0.04, 0.5, 'Generation [GW]', va='center', rotation='vertical', ha='center', fontsize=18)
colors=[beige2,beige3,beige1,brown2, teal2, teal4,yellow,teal3, brown, grey4, grey1]
labels = [label for label in snapshot_base.columns if label != 'Nbattery']  # Exclude 'Nbattery'
patches = [mpatches.Patch(color=color, label=label) for label, color in zip(labels, colors) if label != 'Nbattery']
# Create and place the legend
handles = [mpatches.Patch(color=color, label=label) for label, color in zip(labels, colors)]
fig.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.01), ncol=6, fontsize=14)

plt.subplots_adjust(wspace=0.1)

# Save the figure
plt.savefig(fig_path + 'snapshots.png', dpi=300, bbox_inches='tight')
plt.show()

# print lost load values for all scenarios
lost_load_values = {
    'BASE': snapshot_base['Lost load'].sum(),
    'BASE NZE': snapshot_base_cap['Lost load'].sum(),
    'SC': snapshot_sudden['Lost load'].sum(),
    'SC NZE': snapshot_sudden_cap['Lost load'].sum(),
    'GC': snapshot_gradual['Lost load'].sum(),
    'GC NZE': snapshot_gradual_cap['Lost load'].sum()
}

for scenario, value in lost_load_values.items():
    print(f"Lost load for {scenario}: {value} MWh")
